# Run GSVA

Compute per-sample GSVA scores for every cell type × pathway, using a predefined set of genes from 01. All timepoints are scored together so each sample is enriched against the same fixed background -- this makes the scores comparable across visits.

## 1. Setup
Libraries and pseudobulk reader

In [1]:
suppressPackageStartupMessages({
  library(data.table)
  library(dplyr)
  library(tidyr)
  library(tibble)
  library(GSVA)
  library(DESeq2)
})

# read_pseudobulk_expression()
source("../../00-utilities/functions/r/base-deseq2.R")

## 2. Tissue specific Parameters
The gene list is the all timepoints file per tissue (same `get_filtered_genes_by_label` filter we use for DESeq2: ≥10% of cells per cell type, immunoglobulin / mito / ribo / hemoglobin removed).

In [2]:
tissue <- "bmmc"

pb_dir <- sprintf("../l3-pseudobulk/outputs/%s_l3_raw_gexp_celltypes_per_samplekit/", tissue)
meta_path <- sprintf("../../../manuscript-figures/inputs/metadata/%s_sample_kit_metadata.csv", tissue)
gene_list_path <- sprintf("../../../data/rna/gsva/outputs/%s_l3_gsva_filtered_gene_list.csv", tissue)
gmt_dir <- "../l3-pseudobulk/gmt/"

out_dir <- "../../../data/rna/gsva/results/gsva_results"
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

timepoints_keep <- if (tissue == "pbmc") {
  c("Healthy", "PreTx", "PI2C", "EI", "ASCT60d", "ASCT1y", "ASCT2y")
} else {
  c("Healthy", "PreTx", "EI", "ASCT90d", "ASCT1y", "ASCT2y")
}

## 3. Gene sets
Hallmark, Reactome, ENCODE TF targets, KEGG — read from the same `.gmt` files used in the other analyses.

In [3]:
read_gmt <- function(path) {
  con <- file(path, open = "r")
  on.exit(close(con))
  parts <- strsplit(readLines(con, warn = FALSE), "\t", fixed = TRUE)
  sets <- lapply(parts, function(x) {
    g <- x[-c(1, 2)]
    unique(g[g != ""])
  })
  names(sets) <- toupper(vapply(parts, function(x) x[[1]], character(1)))
  sets
}

gmt_files <- c(
  HALLMARK = "MSigDB_Hallmark_2020.gmt",
  REACTOME = "Reactome_Pathways_2024.gmt",
  ENCODE   = "ENCODE_and_ChEA_Consensus_TFs_from_ChIP-X.gmt",
  KEGG     = "KEGG_2021_Human.gmt"
)

gmt_all <- unlist(
  lapply(names(gmt_files), function(src) {
    g <- read_gmt(file.path(gmt_dir, gmt_files[src]))
    setNames(g, paste0(src, "_", names(g)))
  }),
  recursive = FALSE
)

message("Total gene sets: ", length(gmt_all))

Total gene sets: 2579



## 4. Load metadata, gene list, and pseudobulk counts
`meta_sub` is one row per sample (for the file list). The cell-type vocabulary comes from the *full* metadata — don't pull it from `meta_sub`, since deduplicating by sample would collapse the cell types.

In [4]:
meta <- read.csv(meta_path)

meta_sub <- meta %>%
  filter(label.visitDetails %in% timepoints_keep, keep == "True") %>%
  distinct(sample.sampleKitGuid, .keep_all = TRUE)

celltypes <- unique(meta$aifi_plot_l3)
gene_lists <- fread(gene_list_path) # columns: gene, aifi_plot_l3

message("Cell types: ", length(celltypes), " | samples: ", nrow(meta_sub))

file_list <- paste0(pb_dir, unique(meta_sub$sample.sampleKitGuid), ".csv")
df_list <- read_pseudobulk_expression(file_list)

Cell types: 62 | samples: 67



[1] "Total reading time: 10.133 seconds"
[1] "The length of the list matches the length of the input path."


## 5. Run GSVA per cell type
For a celltype in a sample, subset to that cell type's filtered genes, convert to log-CPM, and score all pathways. Scoring happens once per cell type with every sample present, so the background is fixed across visits.
A cell type is skipped if it has no data or fewer than 10 genes survive the filter.

In [5]:
gsva_long_list <- lapply(celltypes, function(ct) {
  tryCatch(
    {
      # grab this cell type's columns — match the part after "guid:" exactly,
      # so "Treg CD4 Mem" doesn't also catch "Treg CD4 Mem GZMK+"
      ct_list <- lapply(df_list, function(df) {
        ct_of_col <- trimws(sub("^[^:]*:", "", names(df))) # "guid:celltype" -> "celltype"
        cols <- names(df)[ct_of_col == ct]
        if (length(cols) == 0) {
          return(NULL)
        }
        df[, cols, drop = FALSE]
      })
      ct_list <- ct_list[!sapply(ct_list, is.null)]
      if (length(ct_list) == 0) {
        return(NULL)
      }

      count_mat <- as.matrix(do.call(cbind, ct_list))
      colnames(count_mat) <- sub("^([^:]*):.*$", "\\1", colnames(count_mat)) # keep guid only

      # keep only the genes that passed notebook 01's filter for this cell type
      keep_genes <- gene_lists[aifi_plot_l3 == ct, gene]
      count_mat <- count_mat[rownames(count_mat) %in% keep_genes, , drop = FALSE]
      if (nrow(count_mat) < 10) {
        return(NULL)
      }

      # variance-stabilise with DESeq2 — same normalisation family as the DE work.
      # blind = TRUE so the visit design never leaks into the per-sample scores.
      count_mat <- round(count_mat)
      dds <- DESeqDataSetFromMatrix(
        count_mat,
        colData = data.frame(sample = colnames(count_mat)),
        design  = ~1
      )
      expr <- assay(varianceStabilizingTransformation(dds, blind = TRUE, fitType = "local"))

      params <- gsvaParam(expr, gmt_all, minSize = 10, maxSize = 500)
      scores <- gsva(params, verbose = FALSE)

      result <- as.data.frame(t(scores)) %>%
        tibble::rownames_to_column("sample.sampleKitGuid") %>%
        pivot_longer(-sample.sampleKitGuid, names_to = "pathway", values_to = "gsva_score") %>%
        mutate(celltype = ct)

      message("Done: ", ct)
      result
    },
    error = function(e) {
      message("Skipped ", ct, ": ", e$message)
      NULL
    }
  )
})

converting counts to integer mode

Done: gdT

converting counts to integer mode

Done: CD16 Mono Core

converting counts to integer mode

Done: Pre B Prolif

converting counts to integer mode

Done: CD4 T CM

converting counts to integer mode

Done: Pre Mono Core

converting counts to integer mode

Done: CD14 Mono Core

converting counts to integer mode

Done: CD8 T EM2

converting counts to integer mode

Done: CD4 T Naive Core

converting counts to integer mode

Done: NK Effector

converting counts to integer mode

Done: Pre B Light

converting counts to integer mode

Done: Trans B Core

converting counts to integer mode

Done: Prog B Mature

converting counts to integer mode

Done: Treg

converting counts to integer mode

Done: cDC2 ISG+

converting counts to integer mode

Done: CD56dim NK GZMK-

converting counts to integer mode

Done: CLP

converting counts to integer mode

Done: CD8 T Tissue Res

converting counts to integer mode

Done: CD14 Mono ISG+

converting counts to integer

## 6. Write the master sheet
Stack every cell type, attach the visit and subject for each sample, and save.
This file will be used again in 03 (stats) and 04 (plots).

In [6]:
gsva_scores <- bind_rows(gsva_long_list) %>%
  left_join(
    meta_sub %>% select(sample.sampleKitGuid, label.visitDetails, subject.subjectGuid),
    by = "sample.sampleKitGuid"
  )

out_file <- file.path(out_dir, sprintf("gsva_scores_%s.csv", tissue))
fwrite(gsva_scores, out_file)

message("Saved ", nrow(gsva_scores), " rows -> ", out_file)

Saved 5945900 rows -> results/gsva_results/gsva_scores_bmmc.csv



In [7]:
gsva_scores %>% summarise(
    rows      = n(),
    samples   = n_distinct(sample.sampleKitGuid),
    celltypes = n_distinct(celltype),
    pathways  = n_distinct(pathway)
  )

rows,samples,celltypes,pathways
<int>,<int>,<int>,<int>
5945900,67,62,1754
